In [137]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root added:", project_root)

Project root added: /home/junix/marketing-content-verfication


In [138]:
from src.ingestion.load_products import load_products

df = load_products(
    "../data/raw/products.csv"
)

[INFO] Loaded 100 products.


In [139]:
from src.ingestion.load_products import load_products

from src.ingestion.preprocess import (
    preprocess_dataframe
)

df = load_products(
    "../data/raw/products.csv"
)

df = preprocess_dataframe(df)

[INFO] Loaded 100 products.


In [140]:
# from pathlib import Path

# print(Path.cwd())

In [141]:
# import os

# print(os.listdir())

In [142]:
print(df.columns.tolist())

['product_id', 'product_name', 'category', 'brand', 'source', 'price_inr', 'product_description', 'specifications_and_warranty', 'source_url']


In [143]:
import importlib

import src.ingestion.chunking as chunking

importlib.reload(chunking)

<module 'src.ingestion.chunking' from '/home/junix/marketing-content-verfication/src/ingestion/chunking.py'>

In [144]:
from src.ingestion.chunking import (
    create_chunks
)

In [145]:
product_chunks = create_chunks(
    df,
    strategy="product"
)

print(
    "Number of chunks:",
    len(product_chunks)
)

print(
    product_chunks[0]
)

[INFO] Created 100 chunks using 'product' strategy.
Number of chunks: 100
{'product_id': 'Elec-01', 'product_name': 'boAt Airdopes 141 Gen 2 TWS Earbuds', 'category': 'Electronics', 'chunk_type': 'product', 'text': 'Product Name: boAt Airdopes 141 Gen 2 TWS Earbuds\n            Brand: boAt\n            Category: Electronics\n            Price: ₹999\n\n            Description:\n            True wireless earbuds with 48-hour total playback, 4-mic ENx tech for crystal-clear calls, Beast Mode low-latency gaming, and IPX4 sweat resistance.\n\n            Specifications & Warranty:\n            Drivers: 6mm | BT: v5.4 | Playback: 48 hrs (buds+case) | Fast Charge: 10 min = 180 min | IPX4 | Warranty: 1 Year'}


In [146]:
attribute_chunks = create_chunks(
    df,
    strategy="attribute"
)

print(
    "Number of chunks:",
    len(attribute_chunks)
)

print(
    attribute_chunks[0]
)

[INFO] Created 607 chunks using 'attribute' strategy.
Number of chunks: 607
{'product_id': 'Elec-01', 'product_name': 'boAt Airdopes 141 Gen 2 TWS Earbuds', 'category': 'Electronics', 'chunk_type': 'attribute', 'text': 'Product Name: boAt Airdopes 141 Gen 2 TWS Earbuds\n                Brand: boAt\n                Category: Electronics\n\n                Drivers: 6mm'}


In [147]:
print(
    df["specifications_and_warranty"].iloc[0]
)

Drivers: 6mm | BT: v5.4 | Playback: 48 hrs (buds+case) | Fast Charge: 10 min = 180 min | IPX4 | Warranty: 1 Year


In [148]:
print("Product Chunks:", len(product_chunks))
print("Attribute Chunks:", len(attribute_chunks))

Product Chunks: 100
Attribute Chunks: 607


** Experiment 1 complete **

In [149]:
import time
import pandas as pd

from src.embeddings.embed_products import (
    embed_documents
)

In [150]:
import sys

print(sys.executable)

/home/junix/marketing-content-verfication/venv/bin/python


In [187]:
MODEL_NAME = "BAAI/bge-large-en-v1.5"

CHUNKING = "product"

TOP_K = 5

In [188]:
chunks = (
    attribute_chunks
    if CHUNKING == "attribute"
    else product_chunks
)

start = time.time()

embeddings = embed_documents(
    chunks,
    MODEL_NAME
)

embedding_time = time.time() - start

[INFO] Loading model: BAAI/bge-large-en-v1.5


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

[INFO] Generated embeddings for 100 chunks.


In [189]:
print(MODEL_NAME)
print(CHUNKING)
print(TOP_K)
print(embeddings.shape)
print(embedding_time)

BAAI/bge-large-en-v1.5
product
5
(100, 1024)
66.9488935470581


In [190]:
from sklearn.metrics.pairwise import cosine_similarity

In [191]:
query = embeddings[0]

candidate = embeddings[1]

score = cosine_similarity(
    [query],
    [candidate]
)

print(score)

[[0.5846535]]


In [192]:
print(
    "Embedding Shape:",
    embeddings.shape
)

Embedding Shape: (100, 1024)


In [193]:
from src.vectordb.create_faiss import (
    create_index,
    save_index,
    load_index
)

from src.vectordb.search_faiss import (
    search_index
)

In [194]:
index = create_index(
    embeddings
)

[INFO] Added 100 vectors to FAISS.


In [195]:
save_index(
    index,
    "../models/faiss/products.index"
)

[INFO] Saved index: ../models/faiss/products.index


In [196]:
index = load_index(
    "../models/faiss/products.index"
)

[INFO] Loaded index: ../models/faiss/products.index


In [197]:
query_embedding = embeddings[0].reshape(1, -1)

distances, indices = search_index(
    index,
    query_embedding,
    k=5
)

print(indices)
print(distances)

[[ 0  4 19  8 91]]
[[1.0000001 0.8084338 0.7915592 0.745097  0.701005 ]]


In [198]:
for idx in indices[0]:
    print(chunks[idx]["text"])
    print("-" * 50)

Product Name: boAt Airdopes 141 Gen 2 TWS Earbuds
            Brand: boAt
            Category: Electronics
            Price: ₹999

            Description:
            True wireless earbuds with 48-hour total playback, 4-mic ENx tech for crystal-clear calls, Beast Mode low-latency gaming, and IPX4 sweat resistance.

            Specifications & Warranty:
            Drivers: 6mm | BT: v5.4 | Playback: 48 hrs (buds+case) | Fast Charge: 10 min = 180 min | IPX4 | Warranty: 1 Year
--------------------------------------------------
Product Name: Philips TAT1269 True Wireless Earbuds Bluetooth 5.4
            Brand: Philips
            Category: Electronics
            Price: ₹999

            Description:
            Budget TWS earbuds featuring 13mm drivers, Bluetooth 5.4, 40-hour playtime, IPX5 water resistance, and 10-min fast charging via USB-C.

            Specifications & Warranty:
            Drivers: 13mm | BT: 5.4 | Playback: 40 hrs total | Fast Charge: 10 min = 100 min | IPX5 |

In [199]:
import pandas as pd

try:
    experiment_df
except NameError:
    experiment_df = pd.DataFrame()

In [200]:
# experiment_df = pd.DataFrame()

In [201]:
current_result = pd.DataFrame({
    "Embedding Model": [MODEL_NAME],
    "Chunking": [CHUNKING],
    "Embedding Dimension": [embeddings.shape[1]],
    "Chunks": [len(chunks)],
    "Embedding Time (s)": [embedding_time],
    "Top-K": [TOP_K],
    # "Average Similarity": [distances.mean()],
    # "Maximum Similarity": [distances.max()]
})

In [202]:
# experiment_df = experiment_df.drop(
#     columns=[
#         "Average Similarity",
#         "Maximum Similarity"
#     ]
# )

In [203]:
experiment_df = pd.concat(
    [experiment_df, current_result],
    ignore_index=True
)

In [204]:
experiment_df

,Embedding Model,Chunking,Embedding Dimension,Chunks,Embedding Time (s),Top-K
0,BAAI/bge-small-en-v1.5,attribute,384,607,22.353668,5
1,BAAI/bge-small-en-v1.5,product,384,100,17.842308,5
2,BAAI/bge-large-en-v1.5,attribute,1024,607,453.391603,5
3,BAAI/bge-large-en-v1.5,product,1024,100,66.948894,5
